# Task 1 — Public test bằng QLoRA best epoch

Chạy 328 bài public test trên **base model + `best_adapter` đã train ở notebook 05**, không cần Gemini/OpenAI API key. Chọn runtime **GPU** (T4 trở lên).

Notebook đọc `training_summary.json` và `training_config.json` của đúng lượt train, kiểm tra variant/epoch thắng trên **gold validation**, rồi load `best_adapter`. Không lấy checkpoint cuối và không chọn epoch bằng public test. Lượt train đang có trong log notebook 05 chọn **silver_then_gold, epoch 2**; nếu chạy lại training, sửa `TRAINING_RUN` phía dưới.

Quy trình: cài thư viện → mount Drive/clone source → chọn training run → tải public ZIP → preflight toàn bộ prompt → smoke rồi chạy đủ 328 bài. **Push code mới lên branch `task-1-tien` trước khi chạy trên Colab.**

Dự đoán dùng đúng prompt và cách parse điểm khi fine-tune/validation. Chỉ chuẩn hóa tổng điểm bằng tổng sáu chiều như validation; không thêm pipeline rule hoặc prompt của API baseline. Public test không có nhãn nên không tính QWK/MAE. Notebook chỉ tạo file nộp, không tự nộp leaderboard.


In [1]:
%pip -q install "transformers>=4.46,<5" "peft>=0.13,<1" "accelerate>=0.34,<2" "bitsandbytes>=0.45,<1" "pyyaml>=6" "pydantic>=2.5" gdown


In [2]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, zipfile
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > GPU rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))

PROJECT_HOME = Path('/content/drive/MyDrive/LLM/Final Project')
REPO_URL = 'https://github.com/tiendinhquang2104/llm-final-project.git'
REPO_BRANCH = 'task-1-tien'
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
# Clone mới, giữ nguyên các checkout cũ trên Colab.
REPO_DIR = Path('/content') / f'llm-final-project-task1-public-peft-{STAMP}'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
                '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
required = ['scripts/predict_task1_colab.py', 'src/task1/finetune_public.py']
missing = [name for name in required if not (REPO_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Push source mới lên branch {REPO_BRANCH} rồi chạy lại. Thiếu: {missing}')
sys.path = [str(REPO_DIR)] + [p for p in sys.path if p != str(REPO_DIR)]
for module in list(sys.modules):
    if module == 'src' or module.startswith('src.'):
        del sys.modules[module]
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Source:', REPO_DIR, '| commit:', commit)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-80GB
Source: /content/llm-final-project-task1-public-peft-20261019T105934936176Z | commit: 128c5d27847f22105512451afebfcb785b0f9846


## Chọn đúng lượt train

Đường dẫn dưới đây là lượt train đã xuất hiện trong output notebook 05. Đổi sang thư mục output của lần train khác nếu cần; thư mục phải có `training_summary.json`, `training_config.json` và `best_adapter/` đầy đủ, gồm tokenizer.

Variant và epoch được lấy tự động từ summary. Khi checkpoint gốc còn trên Drive, runner so SHA-256 để xác nhận trọng số trong `best_adapter` đúng với checkpoint được chọn. Nếu đã dọn checkpoint gốc, runner dùng bản `best_adapter` do notebook 05 xuất và ghi rõ không đối chiếu được checkpoint.


In [3]:
from src.task1.finetune_public import select_adapter

TRAINING_RUN = (
    PROJECT_HOME / 'data/outputs/task1/qwen25_coder7b_qlora_gold'
    / '20261019T103812050978Z-03a4033c'
)
OUTPUT_ROOT = PROJECT_HOME / 'data/outputs/task1'
selection = select_adapter(TRAINING_RUN)
print('Training run:', TRAINING_RUN)
print('Base model:', selection['model'])
print('Best variant:', selection['selected_variant'], '| best epoch:', selection['selected_epoch'])
print('Gold validation QWK/MAE:', selection['validation_metrics']['qwk'], selection['validation_metrics']['mae'])
print('Adapter:', selection['adapter_dir'])
print('Adapter SHA-256:', selection['adapter_sha256'])
print('Đã đối chiếu checkpoint gốc:', selection['checkpoint_verified'])


Training run: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_gold/20261019T103812050978Z-03a4033c
Base model: Qwen/Qwen2.5-Coder-7B-Instruct
Best variant: gold_only | best epoch: 3
Gold validation QWK/MAE: 0.8401826484018264 1.5
Adapter: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/qwen25_coder7b_qlora_gold/20261019T103812050978Z-03a4033c/best_adapter
Adapter SHA-256: 86de461b03df722f223de64f30c75d37402b702df3fbc76f32034fe1ad6e01aa
Đã đối chiếu checkpoint gốc: True


## Tải public test

Tải mới file ZIP từ Drive; nếu `gdown` bị hạn chế, đặt `DRIVE_ZIP` thành đường dẫn tới bản sao ZIP trong MyDrive. Chỉ đọc `public_test_release.zip` bên trong, không dùng gold/silver làm input cho lượt public test.


In [4]:
import gdown
from src.data.loader import resolve_dataset_root

PUBLIC_ZIP_ID = '1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj'
DRIVE_ZIP = None  # Ví dụ: Path('/content/drive/MyDrive/.../public-test-cham-diem-phan-hoi-files.zip')
WORK_DIR = Path('/content') / f'task1-public-peft-data-{STAMP}'
WORK_DIR.mkdir(parents=True, exist_ok=True)
archive = Path(DRIVE_ZIP) if DRIVE_ZIP else WORK_DIR / 'public-test-cham-diem-phan-hoi-files.zip'
if DRIVE_ZIP is None:
    downloaded = gdown.download(id=PUBLIC_ZIP_ID, output=str(archive), quiet=False)
    if not downloaded:
        raise RuntimeError('Không tải được ZIP. Kiểm tra quyền xem hoặc điền DRIVE_ZIP.')
if not archive.is_file() or not zipfile.is_zipfile(archive):
    raise ValueError(f'Không phải ZIP hợp lệ: {archive}')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

with zipfile.ZipFile(archive) as outer:
    names = [name for name in outer.namelist() if Path(name).name == 'public_test_release.zip']
    if len(names) != 1:
        raise ValueError(f'ZIP ngoài phải chứa đúng một public_test_release.zip; thấy {names}')
    inner_zip = WORK_DIR / 'public_test_release.zip'
    with outer.open(names[0]) as src, inner_zip.open('wb') as dst:
        shutil.copyfileobj(src, dst)
extract_dir = WORK_DIR / 'public'
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(inner_zip) as zf:
    root = extract_dir.resolve()
    if any(not (root / name).resolve().is_relative_to(root) for name in zf.namelist()):
        raise ValueError('ZIP chứa đường dẫn vượt thư mục giải nén')
    bad = zf.testzip()
    if bad:
        raise ValueError(f'Public ZIP lỗi CRC: {bad}')
    zf.extractall(extract_dir)
PUBLIC_ROOT = resolve_dataset_root(extract_dir)
public_source = {
    'drive_file_id': PUBLIC_ZIP_ID if DRIVE_ZIP is None else None,
    'archive_sha256': sha256_file(archive),
    'public_zip_sha256': sha256_file(inner_zip),
    'git_commit': commit,
}
print('Public root:', PUBLIC_ROOT)
print(json.dumps(public_source, indent=2))


Downloading...
From: https://drive.google.com/uc?id=1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj
To: /content/task1-public-peft-data-20261019T105934936176Z/public-test-cham-diem-phan-hoi-files.zip
100%|██████████| 274k/274k [00:00<00:00, 87.3MB/s]

Public root: /content/task1-public-peft-data-20261019T105934936176Z/public/public_test_release
{
  "drive_file_id": "1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj",
  "archive_sha256": "637a033ead19dc2fd5c5a82c859e92637762b09f4da870c2d012f108a9183dad",
  "public_zip_sha256": "67d6b34df702aa482198b7b75039e6ea683a06204ed65c1dc31c291e04ae2366",
  "git_commit": "128c5d27847f22105512451afebfcb785b0f9846"
}


## Preflight và chạy suy luận

Preflight kiểm tra đúng 328 bài không nhãn, sáu đề, tokenizer và độ dài **tất cả** prompt. Mặc định dùng context và số token sinh của `training_config.json`. Nếu có bài vượt context, preflight dừng và liệt kê ID; có thể đặt `MAX_LENGTH` lớn hơn trong giới hạn model, không cắt code.

Cell cuối chạy smoke (một bài mỗi đề và các bài code rỗng) rồi chạy đủ 328 bài trên cùng một model đã load. Checkpoint smoke được dùng tiếp. Chạy lại cell sau khi Colab ngắt sẽ bỏ qua bài hợp lệ; nếu đổi adapter, prompt, source hoặc cấu hình, kết quả có thư mục riêng.


In [5]:
RUNNER = REPO_DIR / 'scripts/predict_task1_colab.py'
MAX_LENGTH = None  # None: giữ max_length lúc train; ví dụ 8192 nếu preflight yêu cầu và GPU đủ bộ nhớ.
RUN_PUBLIC = True

def run_public(*flags):
    command = [sys.executable, '-u', str(RUNNER),
               '--training-run', str(TRAINING_RUN),
               '--public-root', str(PUBLIC_ROOT), '--output-root', str(OUTPUT_ROOT)]
    if MAX_LENGTH is not None:
        command += ['--max-length', str(MAX_LENGTH)]
    subprocess.run(command + list(flags), cwd=REPO_DIR, check=True)

run_public('--preflight')


In [ ]:
if RUN_PUBLIC:
    run_public('--smoke-first')


## File kết quả

`predictions.json` chỉ được tạo khi đủ **328/328 dự đoán hợp lệ**, đúng schema nộp `[{"sample_id": "...", "output": {"rubric": {...}, "total_score": ...}}]`.

`progress.json` ghi variant, best epoch, adapter SHA-256, bài còn thiếu và lỗi. `checkpoints/` lưu từng bài; `invalid_outputs/` lưu phản hồi không parse được, không tự gán điểm 0. `predictions_full.json` giữ raw output và cờ kiểm tra định dạng. Nếu tổng điểm model sinh khác tổng rubric thì tổng được chuẩn hóa như lúc gold validation, đồng thời ghi `n_normalized_total`.

Có thể mở notebook 04 để xem phân bố điểm/rubric, đặt `RUN_DIR` bằng thư mục được in bên dưới. Model fine-tune chỉ sinh rubric, nên phần trạng thái từng câu/rule trong báo cáo sẽ trống.


In [ ]:
reports = []
for path in (OUTPUT_ROOT / 'public/peft').glob('**/progress.json'):
    report = json.loads(path.read_text(encoding='utf-8'))
    if (report.get('adapter_sha256') == selection['adapter_sha256']
            and Path(report.get('adapter_dir', '')) == Path(selection['adapter_dir'])):
        reports.append((path.stat().st_mtime, path, report))
if not reports:
    print('Chưa có lượt suy luận; chạy cell phía trên trước.')
else:
    _, progress_path, progress = max(reports, key=lambda item: item[0])
    RUN_DIR = progress_path.parent
    (RUN_DIR / 'public_source_manifest.json').write_text(
        json.dumps(public_source, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Run:', RUN_DIR)
    print('Best:', progress['selected_variant'], '| epoch:', progress['selected_epoch'])
    print('Tiến độ:', progress['completed'], '/', progress['expected'])
    print('File nộp:', progress['submission_path'] or '(chưa đủ bài hợp lệ)')
    print('Số bài chuẩn hóa tổng:', progress['n_normalized_total'])
    if progress['errors']:
        print('Lỗi:', progress['errors'][:10])
    # Notebook 04 dùng biến này nếu mở trong cùng runtime.
    os.environ['TASK1_PUBLIC_RUN_DIR'] = str(RUN_DIR)
    if progress['submission_path']:
        from src.task1.evaluator import validate_predictions
        from src.data.loader import load_task1
        predictions = json.loads(Path(progress['submission_path']).read_text(encoding='utf-8'))
        expected_ids = sorted(load_task1(PUBLIC_ROOT).by_id())
        assert not validate_predictions(predictions, expected_ids)
        print(json.dumps(predictions[0], ensure_ascii=False, indent=2))


Run: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/public/peft/Qwen-Qwen2.5-Coder-7B-Instruct/20261019T103812050978Z-03a4033c__gold_only__epoch_3/b2fe00163dae0c43
Best: gold_only | epoch: 3
Tiến độ: 328 / 328
File nộp: /content/drive/MyDrive/LLM/Final Project/data/outputs/task1/public/peft/Qwen-Qwen2.5-Coder-7B-Instruct/20261019T103812050978Z-03a4033c__gold_only__epoch_3/b2fe00163dae0c43/predictions.json
Số bài chuẩn hóa tổng: 19
{
  "sample_id": "Class01-Test1-S01",
  "output": {
    "rubric": {
      "compilable": 0,
      "io_format": 0,
      "logic": 0,
      "edge_case": 0,
      "complexity": 0,
      "code_quality": 0
    },
    "total_score": 0
  }
}
